# Privacy clause classifier: experiments

Playground for our own model. Try things here; once something works, copy it into `backend/ml/train.py`.

First build `data/labels.csv` with `backend/ml/label.py`, then fix the labels by hand.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA = Path("..") / "data" / "labels.csv"
df = pd.read_csv(DATA).dropna(subset=["sentence", "label"])
print(len(df), "labelled sentences")
df.sample(10)

In [ ]:
# Is the data balanced? Labels with very few examples will score badly.
df["label"].value_counts().plot.barh(title="Sentences per label")
plt.show()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

X_train, X_test, y_train, y_test = train_test_split(
    df["sentence"], df["label"], test_size=0.2, random_state=42, stratify=df["label"])

model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True),
    LogisticRegression(max_iter=1000, class_weight="balanced"),
)
model.fit(X_train, y_train)
print(classification_report(y_test, model.predict(X_test)))

In [ ]:
# Which labels get confused with each other? Good slide for the pitch.
ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, xticks_rotation="vertical")
plt.show()

In [ ]:
# Try it on your own sentence.
model.predict(["We keep your data for 6 years after your account is closed."])